# Neural Network + Fast Gradient Method (FGM) Evasion Pipeline (ART)

This notebook implements the workflow in your notes (page 1): train a **clean NN**, generate **FGM adversarial samples**, evaluate **clean vs adv**, then perform **adversarial training** and compare **F1 scores**. fileciteturn4file0L1-L8

**Outputs**
- Clean model: accuracy/F1 on clean test and adversarial test
- Adversarially trained model: accuracy/F1 on clean test and adversarial test
- Side-by-side comparison

In [1]:
# If needed:
# !pip install adversarial-robustness-toolbox torch scikit-learn numpy pandas

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from art.defences.trainer import AdversarialTrainer

SEED = 1337
np.random.seed(SEED)
torch.manual_seed(SEED)

C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Load dataset

Edit `LABEL_COL` and `DROP_COLS` to match your CSV.

**Important:** FGM assumes features have known bounds. We'll scale features to **[0, 1]** and set `clip_values=(0.0, 1.0)`.

In [2]:
LABEL_COL = "anomaly"     # change if needed (binary label 0/1)
DROP_COLS = {LABEL_COL}   # add other non-feature columns if needed (timestamps, ids, etc.)

TEST_SIZE = 0.2

def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    y = df[LABEL_COL].astype(int).to_numpy()
    X = df.drop(columns=[c for c in df.columns if c in DROP_COLS], errors="ignore").to_numpy()

    if X.ndim == 1:
        X = X.reshape(-1, 1)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test  = scaler.transform(X_test).astype(np.float32)

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), scaler

# Example:
# X_train, X_test, y_train, y_test, scaler = load_and_prepare(r"CSVs\dataset.csv")

## 2) Define a simple MLP neural network (PyTorch)

You can change hidden sizes/layers without changing the ART pipeline.

In [3]:
class MLP(nn.Module):
    def __init__(self, d_in: int, hidden1: int = 64, hidden2: int = 32, n_classes: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, n_classes),
        )

    def forward(self, x):
        return self.net(x)

## 3) Wrap model with ART `PyTorchClassifier`

FGM needs gradients, so a gradient-capable classifier wrapper is required.

In [4]:
def make_art_classifier(d_in: int, lr: float = 1e-3):
    model = MLP(d_in=d_in)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    art_clf = PyTorchClassifier(
        model=model,
        loss=loss_fn,
        optimizer=optimizer,
        input_shape=(d_in,),
        nb_classes=2,
        clip_values=(0.0, 1.0),  # because we MinMaxScaled features
    )
    return art_clf

## 4) Train clean model and evaluate on clean test

Your notes start with "clean accuracy". fileciteturn4file0L1-L4

In [5]:
def eval_classifier(art_clf: PyTorchClassifier, X: np.ndarray, y: np.ndarray, name: str):
    probs = art_clf.predict(X)              # shape (n,2)
    y_pred = np.argmax(probs, axis=1)

    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    print(f"[{name}] acc={acc:.4f}  f1={f1:.4f}")
    print("confusion matrix:\n", confusion_matrix(y, y_pred))
    return acc, f1

# Example run:
# art_clean = make_art_classifier(d_in=X_train.shape[1])
# art_clean.fit(X_train, y_train, batch_size=128, nb_epochs=20)
# clean_acc, clean_f1 = eval_classifier(art_clean, X_test, y_test, name="Clean model on clean test")

## 5) Generate FGM adversarial samples and evaluate the clean model on them

Your notes: "adv samples → pass to clean model". fileciteturn4file0L1-L4

In [6]:
def make_fgm_attack(art_clf: PyTorchClassifier, eps: float = 0.10):
    # eps is the L_infinity perturbation bound in feature space (here [0,1])
    return FastGradientMethod(estimator=art_clf, eps=eps)

# Example:
# fgm = make_fgm_attack(art_clean, eps=0.10)
# X_test_adv = fgm.generate(x=X_test)
# adv_acc, adv_f1 = eval_classifier(art_clean, X_test_adv, y_test, name="Clean model on FGM test")

## 6) Adversarial training (Robust model)

Two equivalent ways to follow your notes:

### Option A (recommended): ART `AdversarialTrainer`
This matches "Use adversarial trainer → F1 score". fileciteturn4file0L5-L7

### Option B (manual): concat clean + adv and fit (shown after)

In [7]:
def adversarial_train_with_trainer(art_clf: PyTorchClassifier, X_train: np.ndarray, y_train: np.ndarray,
                                  eps: float = 0.10, ratio: float = 0.5,
                                  batch_size: int = 128, nb_epochs: int = 20):
    attack = FastGradientMethod(estimator=art_clf, eps=eps)
    trainer = AdversarialTrainer(classifier=art_clf, attacks=[attack], ratio=ratio)

    # ART trainer forwards training args to underlying estimator where applicable
    trainer.fit(X_train, y_train, batch_size=batch_size, nb_epochs=nb_epochs)
    return trainer.get_classifier()

# Example:
# art_robust = make_art_classifier(d_in=X_train.shape[1])
# art_robust = adversarial_train_with_trainer(art_robust, X_train, y_train, eps=0.10, ratio=0.5, nb_epochs=20)
# eval_classifier(art_robust, X_test, y_test, name="Adv-trained model on clean test")
# X_test_adv = make_fgm_attack(art_robust, eps=0.10).generate(x=X_test)
# eval_classifier(art_robust, X_test_adv, y_test, name="Adv-trained model on FGM test")

## 7) Manual alternative: concat clean + adv then fit

This mirrors your note: "concat clean + ... 'evasion trained'". fileciteturn4file0L2-L4

If you use this option, you don't use `AdversarialTrainer`.

In [8]:
def manual_concat_adversarial_training(art_clf: PyTorchClassifier, X_train: np.ndarray, y_train: np.ndarray,
                                      eps: float = 0.10,
                                      batch_size: int = 128, nb_epochs: int = 20):
    fgm = FastGradientMethod(estimator=art_clf, eps=eps)
    X_adv = fgm.generate(x=X_train)

    X_merged = np.concatenate([X_train, X_adv], axis=0)
    y_merged = np.concatenate([y_train, y_train], axis=0)

    # optional shuffle
    idx = np.random.permutation(len(y_merged))
    X_merged, y_merged = X_merged[idx], y_merged[idx]

    art_clf.fit(X_merged, y_merged, batch_size=batch_size, nb_epochs=nb_epochs)
    return art_clf

# Example:
# art_manual = make_art_classifier(d_in=X_train.shape[1])
# art_manual.fit(X_train, y_train, batch_size=128, nb_epochs=10)  # warm start helps
# art_manual = manual_concat_adversarial_training(art_manual, X_train, y_train, eps=0.10, nb_epochs=20)

## 8) Compare F1 scores (clean vs adversarially trained)

Your notes: "compare F1 scores" and "pass clean to adv trained model dataset". fileciteturn4file0L5-L8

Run this once you have:
- `art_clean` trained
- `art_robust` trained (trainer or manual)

In [ ]:
def compare_models(art_clean: PyTorchClassifier,
                   art_robust: PyTorchClassifier,
                   X_test: np.ndarray,
                   y_test: np.ndarray,
                   eps: float = 0.10):
    print("\n=== CLEAN TEST ===")
    c_acc, c_f1 = eval_classifier(art_clean, X_test, y_test, name="Clean model")
    r_acc, r_f1 = eval_classifier(art_robust, X_test, y_test, name="Adv-trained model")

    fgm_clean = make_fgm_attack(art_clean, eps=eps)
    X_adv = fgm_clean.generate(x=X_test)

    print("\n=== FGM TEST (generated using clean model attack) ===")
    c_acc_a, c_f1_a = eval_classifier(art_clean, X_adv, y_test, name="Clean model")
    r_acc_a, r_f1_a = eval_classifier(art_robust, X_adv, y_test, name="Adv-trained model")

    summary = pd.DataFrame([
        {"model":"clean", "split":"clean", "acc":c_acc, "f1":c_f1},
        {"model":"adv_trained", "split":"clean", "acc":r_acc, "f1":r_f1},
        {"model":"clean", "split":"fgm", "acc":c_acc_a, "f1":c_f1_a},
        {"model":"adv_trained", "split":"fgm", "acc":r_acc_a, "f1":r_f1_a},
    ])
    return summary

# Example end-to-end:
X_train, X_test, y_train, y_test, scaler = load_and_prepare(r"CSVs\dataset.csv")

art_clean = make_art_classifier(d_in=X_train.shape[1])
art_clean.fit(X_train, y_train, batch_size=128, nb_epochs=20)

art_robust = make_art_classifier(d_in=X_train.shape[1])
art_robust = adversarial_train_with_trainer(art_robust, X_train, y_train, eps=0.10, ratio=0.5, nb_epochs=20)

summary_df = compare_models(art_clean, art_robust, X_test, y_test, eps=0.10)
summary_df

ValueError: could not convert string to float: 'CADC0892'

## 9) Tuning notes

- **eps**: start with 0.05 to 0.15 after scaling to [0,1]
- **epochs**: start 10–30
- **ratio** in `AdversarialTrainer`: 0.5 is a common baseline; try 0.25/0.75
- If you get errors about shapes, print `X_train.shape`, `y_train.shape`, and ensure `y` is integer class labels (0/1), not one-hot.